# Computer Vision Assignment 1 — Mango Grading Line
**Run:** Google Colab → Runtime → Run all.

This single notebook contains executable Python + OpenCV code for Tasks 1–10, engineering calculations, synthetic test images, tables and plots. It uses synthetic data for demonstration where the assignment requires self-collected mango images; replace the `DATA` folder with the labelled images to run the same pipeline on real images.


In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, math, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

try:
    from sklearn.metrics import cohen_kappa_score, accuracy_score
except Exception:
    !pip -q install scikit-learn
    from sklearn.metrics import cohen_kappa_score, accuracy_score

BASE=Path("/content/mango_cv_assignment")
DATA=BASE/"data"
OUT=BASE/"outputs"
DATA.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

BELT_SPEED=0.5
FOV_X=400
FOV_Y=300
FPS=8
DECISION_LIMIT=1.2
EJECTOR_DISTANCE=0.6
DEFECT_MM=2.0
STORAGE_TB=2
print("OpenCV:",cv2.__version__)
print("Workspace:",BASE)


## Task 1 — Grading Approaches


In [ ]:
# Task 1: manual, assisted and automated grading
# Synthetic labels are used only to demonstrate the metrics pipeline.
manual_r1=[0,1,1,0,2,2,1,0,1,2]
manual_r2=[0,1,0,0,2,2,1,0,1,2]
manual_r3=[0,1,1,0,2,2,1,0,2,2]
assisted=[0,1,1,0,2,2,1,0,1,2]
automated=[0,1,1,0,2,2,1,0,1,2]
print("Inter-rater kappa R1/R2:", round(cohen_kappa_score(manual_r1,manual_r2),3))
print("Inter-rater kappa R1/R3:", round(cohen_kappa_score(manual_r1,manual_r3),3))
print("Assisted accuracy:", accuracy_score(manual_r1,assisted))
print("Automated accuracy:", accuracy_score(manual_r1,automated))
print("Production demand:", FPS*3600*10, "fruit decisions/10 h")
print("Example worker capacity at 1.5 s/fruit:", round(1/1.5,2), "fruit/s")


## Task 2 — Low-, Mid- and High-Level Pipelines


In [ ]:
# Task 2: synthetic mango + three pipeline depths
def synthetic_mango(seed=0,h=256,w=384):
    rng=np.random.default_rng(seed)
    img=np.zeros((h,w,3),np.uint8); img[:]=(45,80,120)
    cv2.ellipse(img,(w//2,h//2),(100,145),0,0,360,(70,160,190),-1)
    for _ in range(3):
        x=int(rng.integers(w//2-65,w//2+65)); y=int(rng.integers(h//2-90,h//2+90))
        cv2.circle(img,(x,y),int(rng.integers(3,8)),(20,25,30),-1)
    return img

def low_level(img):
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    blur=cv2.GaussianBlur(gray,(5,5),0)
    _,bw=cv2.threshold(blur,70,255,cv2.THRESH_BINARY_INV)
    return bw

def mid_level(img):
    bw=low_level(img)
    n,lab,stats,cent=cv2.connectedComponentsWithStats(bw,8)
    areas=stats[1:,cv2.CC_STAT_AREA] if n>1 else np.array([])
    return {"components":int(np.sum(areas>20)),"largest_area":int(areas.max()) if len(areas) else 0}

def high_level(img):
    m=mid_level(img)
    grade="reject" if m["components"]>4 or m["largest_area"]>1500 else "accept"
    return {"grade":grade,**m}

img=synthetic_mango(1)
print("Low-level dark pixels:", int(np.count_nonzero(low_level(img))))
print("Mid-level features:", mid_level(img))
print("High-level decision:", high_level(img))
plt.figure(figsize=(5,3)); plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()


## Task 3 — RGB/BGR, HSV and Lab


In [ ]:
img=synthetic_mango(2)
hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
print("shape:",img.shape,"dtype:",img.dtype)
M,N,k=img.shape
print("Storage:",M*N*k*8,"bits =",M*N*k,"bytes")
a=np.array([250],dtype=np.uint8); b=np.array([10],dtype=np.uint8)
print("NumPy uint8 + result:",(a+b)[0])
print("cv2.add result:",cv2.add(a,b)[0])
plt.figure(figsize=(6,3)); plt.hist(hsv[...,0].ravel(),bins=36); plt.xlabel("HSV Hue"); plt.ylabel("Pixels"); plt.show()


## Task 4 — Connectivity and Distance Measures


In [ ]:
binary=np.zeros((60,90),np.uint8)
binary[10,10]=255; binary[11,11]=255; binary[12,12]=255
binary[25:30,40:46]=255
binary[40,70]=255; binary[40,71]=255

for conn in [4,8]:
    n,lab,stats,cent=cv2.connectedComponentsWithStats(binary,connectivity=conn)
    print(conn,"-connectivity component count:",n-1)

p=np.array([10,20]); q=np.array([14,23]); d=np.abs(p-q)
print("Euclidean:",np.linalg.norm(p-q))
print("City-block:",d.sum())
print("Chessboard:",d.max())
plt.figure(figsize=(5,3)); plt.imshow(binary,cmap="gray"); plt.axis("off"); plt.show()


## Task 5 — Acquisition, Motion Blur and Line Scan


In [ ]:
for exposure_ms in [0.1,1,4]:
    blur_mm=BELT_SPEED*exposure_ms
    print(f"{exposure_ms} ms -> {blur_mm:.3f} mm motion blur")

line_pitch=FOV_X/2448
line_rate=(BELT_SPEED*1000)/line_pitch
print(f"2448-pixel line pitch = {line_pitch:.3f} mm/line")
print(f"Required line rate = {line_rate:.1f} lines/s")

# Simple rolling-shutter simulation: rows shift progressively
h,w=180,300
base=np.zeros((h,w),np.uint8); cv2.rectangle(base,(100,60),(170,120),255,-1)
skew=np.zeros_like(base)
for y in range(h):
    shift=int(20*y/h)
    skew[y]=np.roll(base[y],shift)
plt.figure(figsize=(7,3))
plt.subplot(1,2,1); plt.imshow(base,cmap="gray"); plt.title("Global exposure"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(skew,cmap="gray"); plt.title("Rolling-shutter skew"); plt.axis("off")
plt.show()


## Task 6 — Spatial Sampling


In [ ]:
rows=[]
for w,h in [(640,480),(1280,1024),(2448,2048)]:
    mmpp=FOV_X/w
    pdef=DEFECT_MM/mmpp
    frame_bytes=w*h*3
    mbps=frame_bytes*FPS/1e6
    gb7=frame_bytes*FPS*60*60*24*7/1e9
    rows.append([f"{w}x{h}",mmpp,pdef,mbps,gb7])
df=pd.DataFrame(rows,columns=["Resolution","mm/pixel","Pixels across 2mm","Raw MB/s","Raw GB/7d"])
display(df)

min_fps=BELT_SPEED/(0.300-0.110)
print("Minimum frame rate for complete 300 mm FOV excluding 110 mm fruit:",min_fps,"fps")

# Downsampling demonstration
img=synthetic_mango(3)
for size in [(640,480),(1280,1024),(2448,2048)]:
    r=cv2.resize(img,size,interpolation=cv2.INTER_AREA)
    print(size,"shape:",r.shape)


## Task 7 — Quantization


In [ ]:
def quantize(img,bits):
    levels=2**bits
    step=256/levels
    return np.clip(np.floor(img/step)*step,0,255).astype(np.uint8)

# Create a subtle 8-bit bruise difference
scene=np.full((180,260),140,dtype=np.uint8)
scene[70:110,110:150]=132

for bits in [8,6,5,4]:
    q=quantize(scene,bits)
    healthy=int(q[30,30]); bruise=int(q[90,130])
    print(bits,"-bit: healthy=",healthy,"bruise=",bruise,"contrast=",abs(healthy-bruise))
    plt.figure(figsize=(3,2)); plt.imshow(q,cmap="gray",vmin=0,vmax=255)
    plt.title(f"{bits}-bit"); plt.axis("off"); plt.show()


## Task 8 — Projection, Perspective and Illumination


In [ ]:
sensor_width=6.4
working_distance=500
focal=sensor_width*working_distance/FOV_X
ratio=working_distance/(working_distance-90)
print("Example focal length:",focal,"mm")
print("Perspective scale ratio at fruit top:",ratio)
print("Approximate scale difference:",(ratio-1)*100,"%")

H,W=240,320
yy,xx=np.mgrid[0:H,0:W]
illum=0.7+0.3*(xx/W)
scene=np.full((H,W),120.0); scene[90:130,140:180]=80
observed=scene*illum
corrected=observed/illum*np.mean(illum)

fig,axs=plt.subplots(1,3,figsize=(10,3))
for ax,a,t in zip(axs,[scene,observed,corrected],["Reflectance","Observed i*r","Flat-field corrected"]):
    ax.imshow(a,cmap="gray"); ax.set_title(t); ax.axis("off")
plt.show()


## Task 9 — Integrated Front-End


In [ ]:
timing={"Acquisition":10,"Transfer":10,"Low-level":20,"Mid-level":25,"Features":25,"High-level":20,"Decision+I/O":10}
total=sum(timing.values())
print("Illustrative total pipeline time:",total,"ms")
print("Decision limit:",DECISION_LIMIT*1000,"ms")
print("Travel time to ejector:",EJECTOR_DISTANCE/BELT_SPEED,"s")

config=pd.DataFrame([
["Budget","0.3–1.3 MP USB rolling shutter","Low","Low-medium"],
["Balanced","1.3 MP global shutter + strobe + calibrated lens","Medium","High"],
["Premium","5 MP/line scan + telecentric + 12-bit","High","Very high"],
["Smart camera","Onboard AI accelerator","Medium-high","High after validation"]
],columns=["Configuration","Acquisition","Cost","Robustness"])
display(config)


## Task 10 — Revised 0.5 mm / 1.0 m/s / NIR Requirement


In [ ]:
new_speed=1.0
new_fps=16
new_defect=0.5
target_px=4
mmpp=new_defect/target_px
required_width=math.ceil(FOV_X/mmpp)
line_rate=(new_speed*1000)/mmpp
print("Target mm/pixel:",mmpp)
print("Required horizontal pixels for 400 mm FOV:",required_width)
print("Required line rate:",line_rate,"lines/s")
for t_ms in [0.1,0.5,1.0]:
    print(t_ms,"ms exposure ->",new_speed*t_ms,"mm blur")

w=required_width; h=2048
frame_bytes=w*h*3
print("Illustrative raw MB/s at 16 fps:",frame_bytes*new_fps/1e6)
print("Illustrative raw TB/7 days:",frame_bytes*new_fps*60*60*24*7/1e12)

strategies=pd.DataFrame([
["A","Retain + ROI","Low change","No NIR","Reject"],
["B","Higher-res + strobe + hood","Moderate","No NIR","Interim"],
["C","RGB + NIR multispectral","High","Yes","Select"]
],columns=["ID","Strategy","Change","NIR","Decision"])
display(strategies)


## Final engineering decisions
1. Original line: choose the balanced global-shutter + strobe + calibrated standard-lens design.
2. Spatial resolution: 1280×1024 is the baseline for the 2 mm defect requirement.
3. Intensity: retain 8-bit lossless colour for the baseline; move to 10/12-bit where measured bruise contrast requires it.
4. Processing: use a three-level pipeline with OpenCV preprocessing, geometric/colour features, and a compact classifier/rule stage.
5. Revised export requirement: redesign to enclosed RGB+NIR sensing with higher spatial sampling, short strobe exposure and encoder synchronisation.
